## Notebook for Supervised Fine Tuning Details

In [1]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
print(f"{tokenizer.chat_template}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-14m")
chat_template = """
    {%- for message in messages %}
        {%- if message['role'] == 'user' %}
            {{- '<|user|>\n' + message['content'] + '\n' }}
        {% elif message['role'] == 'assistant' %}
            {{- '<|assistant|>\n'  + message['content'] + eos_token }}
        {% endif %}
    {%- endfor -%}
    """
tokenizer.chat_template = chat_template

tokenizer_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [3]:
messages = [
    {"role": "user", "content": "write kubectl command to delete the pod px"},
    {"role": "assistant", "content": "kubectl delete pod px"}
]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
print(formatted_prompt)

<|user|>
write kubectl command to delete the pod px

<|assistant|>
kubectl delete pod px<|endoftext|>



In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "EleutherAI/pythia-14m",
    revision="main",
    trust_remote_code="False",
    use_fast=True,
)

chat_template = """{{ bos_token }}{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|system|>
' + message['content'] + '
' }}{% elif message['role'] == 'user' %}{{ '<|user|>
' + message['content'] + '
' }}{% elif message['role'] == 'assistant' %}{% if not loop.last %}{{ '<|assistant|>
'  + message['content'] + eos_token + '
' }}{% else %}{{ '<|assistant|>
'  + message['content'] + eos_token }}{% endif %}{% endif %}{% if loop.last and add_generation_prompt %}{{ '<|assistant|>
' }}{% endif %}{% endfor %}"""
tokenizer.chat_template = chat_template

In [5]:
messages = [
    {"role": "user", "content": "write kubectl command to delete the pod px"},
    {"role": "assistant", "content": "kubectl delete pod px"}
]

In [6]:
formatted_prompt = tokenizer.apply_chat_template(conversation=messages,
        tokenize=False,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=1024,
        add_generation_prompt=False)
print(f"{repr(formatted_prompt)}")

'<|endoftext|><|user|>\nwrite kubectl command to delete the pod px\n<|assistant|>\nkubectl delete pod px<|endoftext|>'


In [7]:
inputs = tokenizer(formatted_prompt, return_tensors="pt")
print(f"Token Count: {len(inputs.input_ids[0])}")
print(f"Tokens: {inputs.input_ids[0]}")

Token Count: 34
Tokens: tensor([    0,    29,    93,  4537, 49651,   187,  6343,   465,   538,   646,
           77,  3923,   281, 11352,   253,  7360,   268,    89,   187,    29,
           93,   515,  5567, 49651,   187,    76,   538,   646,    77, 11352,
         7360,   268,    89,     0])


In [8]:
def visualize_token(tokens: list[int], tokenizer):
    tok_list = []
    for i,token in enumerate(tokens):
        decoded_token = token
        if token != -100:
            decoded_token = tokenizer.decode(token)
        print(f"{token:<5} -> {repr(decoded_token):<15}           ", end='' )
        tok_list.append([f"{token:<5}", f"{repr(decoded_token):<15}"])
        if i % 3 == 1:
            print("\r\n")
visualize_token(inputs.input_ids.tolist()[0], tokenizer)

0     -> '<|endoftext|>'           29    -> '<'                       

93    -> '|'                       4537  -> 'user'                    49651 -> '|>'                      

187   -> '\n'                      6343  -> 'write'                   465   -> ' k'                      

538   -> 'ub'                      646   -> 'ect'                     77    -> 'l'                       

3923  -> ' command'                281   -> ' to'                     11352 -> ' delete'                 

253   -> ' the'                    7360  -> ' pod'                    268   -> ' p'                      

89    -> 'x'                       187   -> '\n'                      29    -> '<'                       

93    -> '|'                       515   -> 'ass'                     5567  -> 'istant'                  

49651 -> '|>'                      187   -> '\n'                      76    -> 'k'                       

538   -> 'ub'                      646   -> 'ect'                     77

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch

model_name ="EleutherAI/pythia-14m"
model = AutoModelForCausalLM.from_pretrained(
                model_name,
                revision=None,
                torch_dtype=torch.bfloat16,
                attn_implementation="eager"
)

config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/53.3M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [11]:
outputs = model(**inputs, use_cache=False)
logits = outputs.logits
print(f"{logits.shape = }")

logits.shape = torch.Size([1, 34, 50304])


In [12]:
labels = torch.tensor([[ -100,  -100, -100, -100, -100,  -100,  -100, -100, -100, -100, -100, -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, -100,  -100,  -100,  -100,  -100,
          76,   538,   646, 77, 11352, 7360, 268, 89, 0]])
print(f"{logits.shape = }")
print(f"{labels.shape = }")
# logits.shape = torch.Size([1, 34, 50304])
# labels.shape = torch.Size([1, 34])

logits.shape = torch.Size([1, 34, 50304])
labels.shape = torch.Size([1, 34])


In [13]:
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()
print(f"{shift_logits.shape = }")
print(f"{shift_labels.shape = }")

# shift_logits.shape = torch.Size([1, 33, 50304])
# shift_labels.shape = torch.Size([1, 33])

shift_logits.shape = torch.Size([1, 33, 50304])
shift_labels.shape = torch.Size([1, 33])


In [14]:
embedding_size = 50304
shift_logits = shift_logits.view(-1, embedding_size)
shift_labels = shift_labels.view(-1)
print(f"{shift_logits.shape = }")
print(f"{shift_labels.shape = }")

# shift_logits.shape = torch.Size([33, 50304])
# shift_labels.shape = torch.Size([33])

shift_logits.shape = torch.Size([33, 50304])
shift_labels.shape = torch.Size([33])


In [15]:
loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")
loss = loss_fct(shift_logits, shift_labels)
loss

# tensor(70.5000, dtype=torch.bfloat16, grad_fn=<NllLossBackward0>)

tensor(71., dtype=torch.bfloat16, grad_fn=<NllLossBackward0>)